In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pathlib

import numpy as np
import toml
import matplotlib.pyplot as plt
import pandas as pd
import pyerrors as pe
import h5py

from compute_observables import (
        compute_average_magnetisation,
        compute_average_energy,
        compute_susceptibility,
        compute_specific_heat,
        compute_binder_cumulant,
    )
from utils.h5_utils import (
    import_lattice,
    import_physical_parameter,
    import_renormalised_energy,
    import_renormalised_magnetisation,
)
from lattice_plots import (
    _generate_lattice
)

# Matplotlib plot parameters
plt.rcParams['text.usetex'] = True
plt.rcParams.update({'font.size':14, 'figure.autolayout':True})



project_name = "temperature30_0-3_1-5_l100_dim2_long"
project_path = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") / project_name 

def import_observable(directory: pathlib.Path, observable_name: str)->int | float:
    f = h5py.File(directory / "results.h5", "r")    
    observable = np.array(f["observables"][observable_name])
    f.close()
    return observable


In [ ]:
from matplotlib import colors
project_name = "above_critical_temp_test"
project_root = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") 
# project_root = pathlib.Path("/home/users/romeroca/capstone-code/projects") 
parameter_combination = 0
project_path = project_root/ project_name / f"parameter-config-{parameter_combination}"
config = toml.load(project_path / "config.toml")
time = 20
lattice = import_lattice(project_path, time)

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

angles = _generate_lattice(time,lattice, config)
y,x = np.indices(angles.shape)

u = np.cos(angles)
v = np.sin(angles)
im = ax2.imshow(
    _generate_lattice(time, import_lattice(project_path,time), config),
      cmap="hsv",
      vmin = 0,
      vmax = 2*np.pi,
      interpolation='bilinear',
      origin = "lower")
norm = colors.Normalize(vmin=0, vmax=2*np.pi)

# q = ax.quiver(x, y, u, v, angles, cmap="hsv", pivot="mid", scale = 30, norm = norm)
step = 1
q_black = ax2.quiver(
    # x[::step, ::step],
    x,
    # y[::step, ::step],
    y,
    # u[::step, ::step],
    u,
    # v[::step, ::step],
    v,
    color="black",
    pivot="mid",
    # scale=50,
    alpha=0.7,
    width=0.003,
    headwidth=3,
    headlength=4,
    headaxislength=3.5,
)
q_chat = ax.quiver(
    x[::step, ::step],
    y[::step, ::step],
    u[::step, ::step],
    v[::step, ::step],
    angles[::step, ::step],
    cmap="hsv",
    norm=norm,
    pivot="mid",
    scale=25,
    width=0.003,
    headwidth=3,
    headlength=4,
    headaxislength=3.5,
)


ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
#q = ax.quiver(x[::step], y[::step], u[::step], v[::step], angles[::step], cmap="hsv", pivot="mid", scale = 20, norm = norm)
cbar = fig.colorbar(q_chat, ax=ax, label="Spin angle (radians)")
cbar.set_ticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
cbar.set_ticklabels(["0", r"$\pi$/2", r"$\pi$", r"3$\pi$/2", r"2$\pi$"])
cbar2 = fig2.colorbar(im, ax=ax2, label="Spin angle (radians)")
cbar2.set_ticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
cbar2.set_ticklabels(["0", r"$\pi$/2", r"$\pi$", r"3$\pi$/2", r"2$\pi$"])

ax.set_aspect('equal')
ax2.set_aspect('equal')
ax.set_title(f"XY model for $J = 1$, $T = {config['physical_settings']['temperature']} / k_B$")
ax2.set_title(f"XY model for $J = 1$, $T = {config['physical_settings']['temperature']} / k_B$")
fig.set_facecolor("#ECEFF4")  
fig2.set_facecolor("#ECEFF4")  
ax.set_facecolor("#ECEFF4")    
ax2.set_facecolor("#ECEFF4")    
fig.savefig("color_arrow_plot.pdf")
fig2.savefig("lattice_arrow_plot.pdf")
plt.close()

In [ ]:
def _generate_lattice(time_step:int, df:pd.DataFrame, config:dict) -> np.array:
    L = config["physical_settings"]["L"]
    dim = config["physical_settings"]["dimension"]
    flat_grid = df.to_numpy().flatten()
    shape = tuple([int(L)] * dim)
    full_dim_grid = flat_grid.reshape(shape)
    return full_dim_grid


In [ ]:
from matplotlib import colors
from tqdm import tqdm
from matplotlib.animation import FuncAnimation
project_name = "rg_test2"
project_root = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") 
# project_root = pathlib.Path("/home/users/romeroca/capstone-code/projects") 
project_path = project_root/ project_name
num_dir = 0
temp_dict = {}
for dir in project_path.iterdir():
    if dir.is_dir():
        temp = import_physical_parameter(project_path /  dir, "temperature")
        temp_dict[dir] = temp
        num_dir += 1
print(temp_dict)
lattice = import_lattice(project_path / "parameter-config-0", 0)
config = toml.load(project_path / "parameter-config-0" / "config.toml")

fig, ax = plt.subplots()

im = ax.imshow(
    _generate_lattice(0, lattice, config),
      cmap="hsv",
      vmin = 0,
      vmax = 2*np.pi,
      interpolation='bilinear',
      origin = "lower")

step = 1
# q_black = ax.quiver(
#     # x[::step, ::step],
#     x,
#     # y[::step, ::step],
#     y,
#     # u[::step, ::step],
#     u,
#     # v[::step, ::step],
#     v,
#     color="black",
#     pivot="mid",
#     # scale=50,
#     alpha=0.7,
#     width=0.003,
#     headwidth=3,
#     headlength=4,
#     headaxislength=3.5,
# )
cbar = fig.colorbar(im, ax=ax, label="Spin angle (radians)")
cbar.set_ticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
cbar.set_ticklabels(["0", r"$\pi$/2", r"$\pi$", r"3$\pi$/2", r"2$\pi$"])

ax.set_aspect('equal')
fig.set_facecolor("#ECEFF4")  
ax.set_facecolor("#ECEFF4")    

frames_ps = 10
nframes = 20 * num_dir

save_path = project_root.parent / "analyze" /"output"/ "vid_dump" 
save_path.parent.mkdir(parents = True, exist_ok = True)
save_path_frame = save_path / "frames"
save_path_frame.mkdir(parents = True, exist_ok = True)
print("save path", save_path)

def update(frame):
    current_project = project_path / f"parameter-config-{frame // 20}"
    config = toml.load(current_project / "config.toml")
    ax.set_title(f"XY model for $J = 1$, $T = {config['physical_settings']['temperature']} / k_B$")
    im.set_array(_generate_lattice(frame % 20, import_lattice(current,frame % 20), config))
    # fig.savefig(save_path_frame / f"frame_{frame:03d}.png", dpi=150)
    return [im]
ani = FuncAnimation(fig, update, frames = tqdm(range(nframes), desc="Rendering"), interval = frames_ps, blit = True, cache_frame_data = False)
ani.save(save_path / f"{project_name}_temp_video.mp4", writer = "ffmpeg", fps =frames_ps, dpi = 150)

save path /home/alvaro/Documents/trinity/year4/capstone/capstone-code/analyze/output/vid_dump


Rendering: 0it [00:00, ?it/s]
/home/alvaro/Documents/trinity/year4/capstone/capstone-code/.pixi/envs/default/lib/python3.13/site-packages/matplotlib/animation.py:1762: UserWarning: Can not start iterating the frames for the initial draw. This can be caused by passing in a 0 length sequence for *frames*.

If you passed *frames* as a generator it may be exhausted due to a previous display or save.
  warnings.warn(
/home/alvaro/Documents/trinity/year4/capstone/capstone-code/.pixi/envs/default/lib/python3.13/site-packages/matplotlib/animation.py:1762: UserWarning: Can not start iterating the frames for the initial draw. This can be caused by passing in a 0 length sequence for *frames*.

If you passed *frames* as a generator it may be exhausted due to a previous display or save.
  warnings.warn(


In [ ]:
from analyze.compute_observables import *
from utils.h5_utils import (
    import_lattice_size)
import pyerrors as pe
project_name = "rg_test2"
project_root = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") 
# project_root = pathlib.Path("/home/users/romeroca/capstone-code/projects") 
parameter_combination = 0
project_path = project_root/ project_name 
magnetisations = []
magnetisations_obs = []
bs = [2]

for i in range(15):
    for b in bs:
        mag_arr = import_renormalised_magnetisation(project_path / f"parameter-config-{i}", b)
        N = import_lattice_size(project_path / f"parameter-config-{i}") / (b**2)
        x_mag = mag_arr[:,0] / N
        y_mag = mag_arr[:,1] / N
        magnetisation = np.sqrt(x_mag**2 + y_mag**2) 
        mag_obs = pe.Obs([magnetisation[1000:]], ["magnetisation"])
        mag_obs.gamma_method()
        magnetisations.append(np.average(magnetisation[1000:]))
        magnetisations_obs.append(mag_obs.value)
        plt.plot(magnetisation, label=f"b = {b}", alpha=1 / b)
    plt.title(f"parameter combination {i}")
    plt.legend()
    plt.show()
    plt.close()

In [ ]:
21 //20